In [ ]:
MODEL = "bert-base-uncased"
TASKS = ["sst2"]
SEEDS = [0, 1, 2]
OUT_ROOT = "/kaggle/working/ckpt"
EPOCHS = {"sst2": 3, "cola": 3, "mrpc": 5}
LR, BATCH, MAX_LEN, WARMUP, WD = 2e-5, 32, 128, 0.1, 0.01
DEV_FRAC = 0.05          

import json, random, time
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", dev_, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}
tok = AutoTokenizer.from_pretrained(MODEL)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def prep(task):
    try:
        ds = load_dataset("nyu-mll/glue", task)      # current canonical repo id
    except Exception as e:
        print("nyu-mll/glue failed (", type(e).__name__, "), retrying with legacy name 'glue'")
        ds = load_dataset("glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else \
               tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=DEV_FRAC, seed=0)      # fixed across seeds
    parts = {"train": split["train"], "dev": split["test"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        d = d.remove_columns([c for c in d.column_names if c not in cols])
        out[n] = d
    # fixed 5000-example train subsample for measuring the train-side loss/accuracy
    n_sub = min(5000, len(out["train"]))
    idx = np.random.RandomState(0).choice(len(out["train"]), n_sub, replace=False)
    out["train_sub"] = out["train"].select(sorted(idx.tolist()))
    return out


def collate(batch):
    labels = torch.tensor([b["label"] for b in batch])
    feats = [{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch]
    enc = tok.pad(feats, return_tensors="pt")
    enc["labels"] = labels
    return enc


@torch.no_grad()
def predict(model, ds, bs=128):
    model.eval()
    logits, labels = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            out = model(**b).logits
        logits.append(out.float().cpu()); labels.append(lab)
    return torch.cat(logits).numpy(), torch.cat(labels).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    t = torch.tensor(logits); l = torch.tensor(labels)
    loss = torch.nn.functional.cross_entropy(t, l).item()
    m = {"acc": float((p == labels).mean()), "loss": loss}
    if task == "cola": m["mcc"] = float(matthews_corrcoef(labels, p))
    if task == "mrpc": m["f1"] = float(f1_score(labels, p))
    return m


for task in TASKS:
    data = prep(task)
    print(f"\n##### {MODEL} / {task}: train={len(data['train'])} dev={len(data['dev'])} val={len(data['val'])}")
    for seed in SEEDS:
        run_dir = f"{OUT_ROOT}/{MODEL.split('/')[-1]}_{task}_seed{seed}"
        if os.path.exists(f"{run_dir}/done.json"):
            print("skip (done):", run_dir); continue
        os.makedirs(run_dir, exist_ok=True)
        set_seed(seed)
        t0 = time.time()
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(dev_)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        loader = DataLoader(data["train"], batch_size=BATCH, shuffle=True, collate_fn=collate,
                            generator=torch.Generator().manual_seed(seed))
        total = len(loader) * EPOCHS[task]
        sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
        scaler = torch.cuda.amp.GradScaler(enabled=dev_.type == "cuda")

        for ep in range(EPOCHS[task]):
            model.train(); run_loss = 0.0
            for b in loader:
                b = {k: v.to(dev_) for k, v in b.items()}
                with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
                    loss = model(**b).loss
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                run_loss += loss.item()
            lv, yv = predict(model, data["val"])
            print(f"  seed {seed} epoch {ep+1}: train_loss={run_loss/len(loader):.4f} "
                  f"val_acc={metrics(task, lv, yv)['acc']:.4f}")

        
        res = {"model": MODEL, "task": task, "seed": seed, "epochs": EPOCHS[task],
               "lr": LR, "batch": BATCH, "max_len": MAX_LEN, "warmup": WARMUP,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "train_minutes": (time.time() - t0) / 60}
        for name in ["train_sub", "dev", "val"]:
            lg, lb = predict(model, data[name])
            np.save(f"{run_dir}/{name}_logits.npy", lg); np.save(f"{run_dir}/{name}_labels.npy", lb)
            res[name] = metrics(task, lg, lb)
        res["gen_gap_loss"] = res["val"]["loss"] - res["train_sub"]["loss"]   # val loss - train loss
        res["gen_gap_acc"] = res["train_sub"]["acc"] - res["val"]["acc"]      # train acc - val acc
        model.save_pretrained(run_dir); tok.save_pretrained(run_dir)
        json.dump(res, open(f"{run_dir}/done.json", "w"), indent=2)
        print("  saved:", run_dir, {k: round(v, 4) for k, v in res["val"].items()})
        del model, opt; torch.cuda.empty_cache()

print("\nALL DONE")

Device: cuda Tesla T4


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

... (26 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  seed 2 epoch 1: train_loss=0.2523 val_acc=0.9128
  seed 2 epoch 2: train_loss=0.1162 val_acc=0.9278
  seed 2 epoch 3: train_loss=0.0753 val_acc=0.9209


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved: /kaggle/working/ckpt/bert-base-uncased_sst2_seed2 {'acc': 0.9209, 'loss': 0.2855}

ALL DONE
